# SE4050 – Deep Learning (2026)
## Smartphone-Based Recognition of Human Activities and Postural Transitions (HAPT)
### Model: Stacked Bidirectional Long Short-Term Memory (Bi-LSTM) Network

**Module:** SE4050 – Deep Learning  
**Course:** BSc (Hons) in Information Technology  
**Dataset:** Smartphone-Based Recognition of Human Activities and Postural Transitions (UCI HAPT - ID: 341)  
**Target:** 12-Class Classification (6 Basic Physical Activities + 6 Postural Transitions)




In [ ]:
import os
import sys
import time
import random
import json
import ssl
import zipfile
import urllib.request
import subprocess
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import torch
import torch.nn as nn
from torch.optim import Adam
from torch.optim.lr_scheduler import ReduceLROnPlateau
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import (
    classification_report, confusion_matrix, accuracy_score,
    f1_score, precision_score, recall_score, matthews_corrcoef, roc_auc_score
)

# Resolve project directories dynamically (works locally, in VS Code, and in Cloud)
BASE_DIR = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA_DIR = BASE_DIR / "data"
RAW_DATA_DIR = DATA_DIR / "raw"
PROCESSED_DATA_DIR = DATA_DIR / "processed"
MODELS_DIR = BASE_DIR / "models"
OUTPUTS_DIR = BASE_DIR / "outputs"
FIGURES_DIR = OUTPUTS_DIR / "figures"
METRICS_DIR = OUTPUTS_DIR / "metrics"

for d in [RAW_DATA_DIR, PROCESSED_DATA_DIR, MODELS_DIR, FIGURES_DIR, METRICS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

# Set random seed for 100% scientific reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

# Detect hardware acceleration (Apple Silicon MPS -> NVIDIA CUDA -> CPU)
if hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
    device = torch.device("mps")
    print("[INFO] Hardware Acceleration: Apple Silicon GPU (MPS) detected!")
elif torch.cuda.is_available():
    device = torch.device("cuda")
    print(f"[INFO] Hardware Acceleration: NVIDIA GPU ({torch.cuda.get_device_name(0)}) detected!")
else:
    device = torch.device("cpu")
    print("[INFO] Using CPU for computation.")

def synchronize_device(device):
    """Ensures GPU/MPS kernels finish execution for accurate latency benchmarking."""
    if device.type == "cuda":
        torch.cuda.synchronize()
    elif device.type == "mps" and hasattr(torch.mps, "synchronize"):
        torch.mps.synchronize()


---
## 1. Automated Dataset Acquisition

The dataset is obtained directly from the **UCI Machine Learning Repository** (ID: 341).
* **Volunteers:** 30 participants (aged 19–48) wearing a waist-mounted Samsung Galaxy S II smartphone.
* **Sensors:** Tri-axial Accelerometer (total linear acceleration $X, Y, Z$) and Tri-axial Gyroscope (angular velocity $X, Y, Z$) sampled at **50 Hz**.
* **Target:** 12 distinct classes (6 basic steady-state physical activities + 6 transient postural shifts).

The code below automatically downloads and extracts the raw dataset if not already cached.


In [ ]:
DATA_URL = "https://archive.ics.uci.edu/static/public/341/smartphone+based+recognition+of+human+activities+and+postural+transitions.zip"
ZIP_PATH = RAW_DATA_DIR / "hapt_dataset.zip"
EXTRACT_DIR = RAW_DATA_DIR / "HAPT_Data_Set"

def download_and_extract_dataset():
    if EXTRACT_DIR.exists() and any(EXTRACT_DIR.iterdir()):
        print(f"[INFO] Dataset already extracted at: {EXTRACT_DIR}")
        return
    if not ZIP_PATH.exists():
        print(f"[INFO] Downloading UCI HAPT dataset from:\n  {DATA_URL}")
        try:
            ctx = ssl._create_unverified_context()
            req = urllib.request.Request(DATA_URL, headers={"User-Agent": "Mozilla/5.0"})
            with urllib.request.urlopen(req, context=ctx, timeout=60) as resp, open(ZIP_PATH, "wb") as f:
                f.write(resp.read())
            print("[SUCCESS] Download completed via urllib.")
        except Exception as e:
            print(f"[WARN] urllib download failed ({e}), falling back to curl...")
            subprocess.run(["curl", "-L", "-k", "-o", str(ZIP_PATH), DATA_URL], check=True)
            print("[SUCCESS] Download completed via curl.")
    
    print(f"[INFO] Extracting {ZIP_PATH.name} ...")
    with zipfile.ZipFile(ZIP_PATH, "r") as zf:
        zf.extractall(EXTRACT_DIR)
    print(f"[SUCCESS] Dataset extracted to: {EXTRACT_DIR}")

download_and_extract_dataset()


---
## 2. Sliding Window Segmentation & Leakage-Free Subject Partitioning

### Crucial Methodological Decision: Anti-Leakage Subject-Based Splitting
In human activity recognition, **random sample splitting is a critical error**. Because consecutive sliding windows overlap by 50%, random shuffling leaks temporal data from the same subject into both training and testing sets, yielding deceptively inflated test scores that collapse on new people.

To ensure genuine real-world generalization, we partition data by **Volunteer IDs**:
* **Training Set:** Volunteers 1 to 21 ($\approx 68\%$, 7,414 windows)
* **Validation Set:** Volunteers 22 to 25 ($\approx 14\%$, 1,578 windows)
* **Unseen Test Set:** Volunteers 26 to 30 ($\approx 18\%$, 1,977 windows, strictly reserved for final evaluation)

### Window Parameters:
* **Window Duration ($T$):** 128 timesteps $= 128 / 50\text{ Hz} = 2.56\text{ seconds}$ (capturing full gait cycles).
* **Stride ($S$):** 64 timesteps ($50\%$ overlap).
* **Sensor Channels ($C=6$):** $[Acc_X, Acc_Y, Acc_Z, Gyro_X, Gyro_Y, Gyro_Z]$.
* **Standardization:** Z-Score normalization ($\mu, \sigma$) is calculated **strictly on the Training Set** and then applied downstream to prevent data snooping.


In [ ]:
ACTIVITY_NAMES = {
    1: "WALKING", 2: "WALKING_UPSTAIRS", 3: "WALKING_DOWNSTAIRS",
    4: "SITTING", 5: "STANDING", 6: "LAYING",
    7: "STAND_TO_SIT", 8: "SIT_TO_STAND", 9: "SIT_TO_LIE",
    10: "LIE_TO_SIT", 11: "STAND_TO_LIE", 12: "LIE_TO_STAND"
}
CLASS_NAMES = [ACTIVITY_NAMES[i] for i in range(1, 13)]

def load_or_process_data(window_size=128, stride=64, force_recompute=False):
    cache_files = [
        PROCESSED_DATA_DIR / "X_train.npy", PROCESSED_DATA_DIR / "y_train.npy",
        PROCESSED_DATA_DIR / "X_val.npy", PROCESSED_DATA_DIR / "y_val.npy",
        PROCESSED_DATA_DIR / "X_test.npy", PROCESSED_DATA_DIR / "y_test.npy",
        PROCESSED_DATA_DIR / "train_mean.npy", PROCESSED_DATA_DIR / "train_std.npy"
    ]
    meta_file = PROCESSED_DATA_DIR / "metadata.json"

    cache_valid = False
    if not force_recompute and all(f.exists() for f in cache_files) and meta_file.exists():
        try:
            with open(meta_file, "r") as f:
                meta = json.load(f)
            if meta.get("window_size") == window_size and meta.get("stride") == stride:
                cache_valid = True
        except Exception:
            cache_valid = False

    if cache_valid:
        print(f"[INFO] Loading validated cached data (window={window_size}, stride={stride}) from {PROCESSED_DATA_DIR} ...")
        return (
            np.load(PROCESSED_DATA_DIR / "X_train.npy"), np.load(PROCESSED_DATA_DIR / "y_train.npy"),
            np.load(PROCESSED_DATA_DIR / "X_val.npy"), np.load(PROCESSED_DATA_DIR / "y_val.npy"),
            np.load(PROCESSED_DATA_DIR / "X_test.npy"), np.load(PROCESSED_DATA_DIR / "y_test.npy")
        )

    candidates = list(EXTRACT_DIR.glob("**/labels.txt"))
    assert len(candidates) > 0, f"labels.txt not found in {EXTRACT_DIR}"
    raw_dir = candidates[0].parent

    labels_df = pd.read_csv(raw_dir / "labels.txt", sep=r"\s+", header=None,
                            names=["exp_id", "user_id", "act_id", "start_idx", "end_idx"])

    windows, labels, subjects = [], [], []

    for (exp_id, user_id), group in labels_df.groupby(["exp_id", "user_id"]):
        acc_file = raw_dir / f"acc_exp{exp_id:02d}_user{user_id:02d}.txt"
        gyro_file = raw_dir / f"gyro_exp{exp_id:02d}_user{user_id:02d}.txt"
        if not acc_file.exists() or not gyro_file.exists():
            continue
        acc_data = np.loadtxt(acc_file)
        gyro_data = np.loadtxt(gyro_file)
        min_len = min(len(acc_data), len(gyro_data))
        sensor_data = np.hstack([acc_data[:min_len], gyro_data[:min_len]])

        for _, row in group.iterrows():
            act_id = int(row["act_id"]) - 1
            start = int(row["start_idx"]) - 1
            end = int(row["end_idx"])
            dur = end - start
            if dur < window_size:
                if dur >= window_size // 2:
                    sub_seq = sensor_data[start:end]
                    padded = np.pad(sub_seq, ((0, window_size - dur), (0, 0)), mode="edge")
                    windows.append(padded)
                    labels.append(act_id)
                    subjects.append(user_id)
                continue
            for w_start in range(start, end - window_size + 1, stride):
                windows.append(sensor_data[w_start:w_start + window_size])
                labels.append(act_id)
                subjects.append(user_id)

    X = np.array(windows, dtype=np.float32)
    y = np.array(labels, dtype=np.int64)
    subj = np.array(subjects, dtype=np.int32)

    train_m = (subj >= 1) & (subj <= 21)
    val_m = (subj >= 22) & (subj <= 25)
    test_m = (subj >= 26) & (subj <= 30)

    X_train, y_train = X[train_m], y[train_m]
    X_val, y_val = X[val_m], y[val_m]
    X_test, y_test = X[test_m], y[test_m]

    # Z-Score Standardization (Fitted exclusively on X_train to prevent leakage)
    mean = np.mean(X_train, axis=(0, 1), keepdims=True)
    std = np.std(X_train, axis=(0, 1), keepdims=True)
    std[std == 0] = 1.0

    X_train = (X_train - mean) / std
    X_val = (X_val - mean) / std
    X_test = (X_test - mean) / std

    np.save(PROCESSED_DATA_DIR / "X_train.npy", X_train)
    np.save(PROCESSED_DATA_DIR / "y_train.npy", y_train)
    np.save(PROCESSED_DATA_DIR / "X_val.npy", X_val)
    np.save(PROCESSED_DATA_DIR / "y_val.npy", y_val)
    np.save(PROCESSED_DATA_DIR / "X_test.npy", X_test)
    np.save(PROCESSED_DATA_DIR / "y_test.npy", y_test)
    np.save(PROCESSED_DATA_DIR / "train_mean.npy", mean)
    np.save(PROCESSED_DATA_DIR / "train_std.npy", std)

    with open(meta_file, "w") as f:
        json.dump({
            "window_size": window_size, "stride": stride, "num_classes": 12,
            "train_samples": int(len(y_train)), "val_samples": int(len(y_val)), "test_samples": int(len(y_test))
        }, f, indent=2)

    return X_train, y_train, X_val, y_val, X_test, y_test

X_train, y_train, X_val, y_val, X_test, y_test = load_or_process_data()

print(f"[TENSOR SHAPES]")
print(f"X_train: {X_train.shape} | y_train: {y_train.shape}")
print(f"X_val:   {X_val.shape}   | y_val:   {y_val.shape}")
print(f"X_test:  {X_test.shape}  | y_test:  {y_test.shape}")


---
## 3. Exploratory Data Analysis (EDA)

We explore the fundamental properties of the segmented sensor dataset:
1. **Severe Class Imbalance:** Comparing the frequency of steady-state activities (walking, sitting, laying) against rare postural shifts (stand-to-sit, lie-to-sit).
2. **Signal Waveforms:** Visualizing tri-axial accelerometer and gyroscope patterns for periodic movement vs. transient transitions.


In [ ]:
# 1. Class Distribution Bar Plot
train_counts = np.bincount(y_train, minlength=12)
val_counts = np.bincount(y_val, minlength=12)
test_counts = np.bincount(y_test, minlength=12)

df_dist = pd.DataFrame({
    "Activity": CLASS_NAMES,
    "Train (Subj 1-21)": train_counts,
    "Val (Subj 22-25)": val_counts,
    "Test (Subj 26-30)": test_counts
})

plt.figure(figsize=(14, 6))
df_dist.set_index("Activity").plot(kind="bar", stacked=True, figsize=(14, 6), colormap="viridis", edgecolor="black")
plt.title("Class Distribution Across Subject-Wise Splits (HAPT 12 Classes)", fontsize=14, fontweight="bold")
plt.xlabel("Activity / Postural Transition Class", fontsize=12)
plt.ylabel("Number of 2.56s Windows", fontsize=12)
plt.xticks(rotation=45, ha="right", fontsize=10)
plt.grid(axis="y", linestyle=":", alpha=0.7)
plt.tight_layout()
plt.savefig(FIGURES_DIR / "class_distribution.png", dpi=300)
plt.show()

# 2. Multi-Channel Signal Waveform Sample (Walking)
sample_window = X_train[0]
time_axis = np.linspace(0, 2.56, 128)

fig, axes = plt.subplots(2, 1, figsize=(14, 6), sharex=True)
axes[0].plot(time_axis, sample_window[:, 0], label="Acc X", color="crimson")
axes[0].plot(time_axis, sample_window[:, 1], label="Acc Y", color="forestgreen")
axes[0].plot(time_axis, sample_window[:, 2], label="Acc Z", color="dodgerblue")
axes[0].set_title(f"Sample Sensor Signal: {CLASS_NAMES[y_train[0]]} (Tri-axial Accelerometer)", fontsize=13, fontweight="bold")
axes[0].set_ylabel("Normalized Accel", fontsize=11)
axes[0].legend(loc="upper right")
axes[0].grid(True, linestyle=":", alpha=0.6)

axes[1].plot(time_axis, sample_window[:, 3], label="Gyro X", color="orange")
axes[1].plot(time_axis, sample_window[:, 4], label="Gyro Y", color="purple")
axes[1].plot(time_axis, sample_window[:, 5], label="Gyro Z", color="teal")
axes[1].set_title(f"Sample Sensor Signal: {CLASS_NAMES[y_train[0]]} (Tri-axial Gyroscope)", fontsize=13, fontweight="bold")
axes[1].set_xlabel("Time (seconds)", fontsize=11)
axes[1].set_ylabel("Normalized Angular Vel", fontsize=11)
axes[1].legend(loc="upper right")
axes[1].grid(True, linestyle=":", alpha=0.6)

plt.tight_layout()
plt.savefig(FIGURES_DIR / "sensor_signals_sample.png", dpi=300)
plt.show()


### Analysis of EDA Findings:
* **Severe Class Imbalance:** Basic activities (classes 1–6) account for over 85% of samples, whereas postural transitions (classes 7–12) represent less than 15%. Without intervention, standard Cross-Entropy Loss would ignore transition classes.
* **Inverse Frequency Class Weights:**
  To solve this imbalance, we compute:
  $$w_c = \frac{N_{\text{total}}}{C \times N_c}$$
  and inject them directly into the CrossEntropyLoss criterion.


In [ ]:
class HAPTDataset(Dataset):
    def __init__(self, X: np.ndarray, y: np.ndarray):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.long)
    def __len__(self):
        return len(self.y)
    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

def compute_class_weights(y_train, num_classes=12):
    counts = np.bincount(y_train, minlength=num_classes)
    total = len(y_train)
    weights = [total / (num_classes * c) if c > 0 else 1.0 for c in counts]
    weights = np.array(weights, dtype=np.float32)
    weights = weights / np.mean(weights)
    return torch.tensor(weights, dtype=torch.float32)

class_weights = compute_class_weights(y_train, num_classes=12).to(device)

print("[INVERSE CLASS WEIGHTS]")
for name, w in zip(CLASS_NAMES, class_weights):
    print(f"  {name:<22}: weight = {w.item():.3f}")

BATCH_SIZE = 64
train_loader = DataLoader(HAPTDataset(X_train, y_train), batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(HAPTDataset(X_val, y_val), batch_size=BATCH_SIZE, shuffle=False)
test_loader = DataLoader(HAPTDataset(X_test, y_test), batch_size=BATCH_SIZE, shuffle=False)

print(f"\nDataLoaders: Train {len(train_loader)} batches | Val {len(val_loader)} batches | Test {len(test_loader)} batches")


---
## 4. Deep Learning Model Architecture: Stacked Bidirectional LSTM

### Mathematical Justification:
Standard RNNs suffer from vanishing gradients over $T=128$ timesteps. The **Bidirectional LSTM** solves this via:
1. **Constant Error Carousel (CEC):** Additive cell state updates $C_t = f_t \odot C_{t-1} + i_t \odot \tilde{C}_t$ preserve error gradients across all 128 timesteps.
2. **Bidirectional Temporal Flow:** Postural transitions (e.g. `STAND_TO_SIT`) are characterized by how the movement begins and how the subject settles. Concatenating forward ($\overrightarrow{h_t}$) and backward ($\overleftarrow{h_t}$) hidden states captures this full context.


In [ ]:
class ActivityLSTM(nn.Module):
    def __init__(self, input_dim=6, hidden_dim_1=128, hidden_dim_2=64, fc_dim=64, num_classes=12, dropout_rate=0.3):
        super(ActivityLSTM, self).__init__()
        self.lstm1 = nn.LSTM(input_size=input_dim, hidden_size=hidden_dim_1, num_layers=1,
                             batch_first=True, bidirectional=True)
        self.dropout1 = nn.Dropout(dropout_rate)
        self.lstm2 = nn.LSTM(input_size=hidden_dim_1 * 2, hidden_size=hidden_dim_2, num_layers=1,
                             batch_first=True, bidirectional=True)
        self.bn = nn.BatchNorm1d(hidden_dim_2 * 2)
        self.fc = nn.Linear(hidden_dim_2 * 2, fc_dim)
        self.relu = nn.ReLU()
        self.dropout2 = nn.Dropout(0.4)
        self.classifier = nn.Linear(fc_dim, num_classes)

    def forward(self, x):
        out, _ = self.lstm1(x)
        out = self.dropout1(out)
        out, (h_n, c_n) = self.lstm2(out)
        forward_hidden = h_n[0, :, :]
        backward_hidden = h_n[1, :, :]
        context = torch.cat([forward_hidden, backward_hidden], dim=1)
        context = self.bn(context)
        rep = self.relu(self.fc(context))
        rep = self.dropout2(rep)
        return self.classifier(rep)

model = ActivityLSTM(input_dim=6, hidden_dim_1=128, hidden_dim_2=64, num_classes=12).to(device)
param_count = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(model)
print(f"\nTotal Trainable Parameters: {param_count:,}")


---
## 5. Model Training & Validation Protocol

* **Loss Function:** Weighted Cross-Entropy (`nn.CrossEntropyLoss(weight=class_weights)`)
* **Optimizer:** Adam ($\eta = 10^{-3}$, weight decay $= 10^{-4}$)
* **Learning Rate Scheduler:** `ReduceLROnPlateau(mode='max', factor=0.5, patience=4)` monitoring validation **Macro-F1**.
* **Gradient Clipping:** Max norm of $5.0$.
* **Early Stopping & Checkpointing:** Automatically preserves the best weights in `models/best_lstm_model.pth`.


In [ ]:
CHECKPOINT_PATH = MODELS_DIR / "best_lstm_model.pth"
EPOCHS = 25
LEARNING_RATE = 0.001
PATIENCE = 10

criterion = nn.CrossEntropyLoss(weight=class_weights)
optimizer = Adam(model.parameters(), lr=LEARNING_RATE, weight_decay=1e-4)
scheduler = ReduceLROnPlateau(optimizer, mode="max", factor=0.5, patience=4)

history = {"train_loss": [], "val_loss": [], "train_acc": [], "val_acc": [], "val_macro_f1": []}

best_val_f1 = -1.0
best_epoch = 0
patience_counter = 0

print(f"Starting training on {device} for {EPOCHS} epochs...")

for epoch in range(1, EPOCHS + 1):
    t0 = time.time()
    # Training
    model.train()
    running_weighted_loss, total_weight, all_preds, all_targets = 0.0, 0.0, [], []
    for bx, by in train_loader:
        bx, by = bx.to(device), by.to(device)
        optimizer.zero_grad()
        logits = model(bx)
        loss = criterion(logits, by)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0)
        optimizer.step()

        batch_weight_sum = class_weights[by].sum().item()
        running_weighted_loss += loss.item() * batch_weight_sum
        total_weight += batch_weight_sum

        all_preds.extend(torch.argmax(logits, dim=1).detach().cpu().numpy())
        all_targets.extend(by.detach().cpu().numpy())
    train_loss = running_weighted_loss / (total_weight if total_weight > 0 else 1.0)
    train_acc = accuracy_score(all_targets, all_preds) * 100.0

    # Validation
    model.eval()
    val_loss_sum, val_weight, v_preds, v_targets = 0.0, 0.0, [], []
    with torch.no_grad():
        for bx, by in val_loader:
            bx, by = bx.to(device), by.to(device)
            logits = model(bx)
            loss = criterion(logits, by)

            batch_weight_sum = class_weights[by].sum().item()
            val_loss_sum += loss.item() * batch_weight_sum
            val_weight += batch_weight_sum

            v_preds.extend(torch.argmax(logits, dim=1).cpu().numpy())
            v_targets.extend(by.cpu().numpy())
    val_loss = val_loss_sum / (val_weight if val_weight > 0 else 1.0)
    val_acc = accuracy_score(v_targets, v_preds) * 100.0
    val_macro_f1 = f1_score(v_targets, v_preds, average="macro") * 100.0
    scheduler.step(val_macro_f1)

    elapsed = time.time() - t0
    history["train_loss"].append(train_loss)
    history["val_loss"].append(val_loss)
    history["train_acc"].append(train_acc)
    history["val_acc"].append(val_acc)
    history["val_macro_f1"].append(val_macro_f1)

    print(f"Epoch [{epoch:02d}/{EPOCHS:02d}] ({elapsed:.1f}s) | "
          f"Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.2f}% | "
          f"Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.2f}% | Val Macro-F1: {val_macro_f1:.2f}%")

    if val_macro_f1 > best_val_f1:
        best_val_f1 = val_macro_f1
        best_epoch = epoch
        patience_counter = 0
        torch.save(model.state_dict(), CHECKPOINT_PATH)
        print(f"  --> [SAVED] Best checkpoint (Val Macro-F1: {best_val_f1:.2f}%)")
    else:
        patience_counter += 1
        if patience_counter >= PATIENCE:
            print("\n[EARLY STOPPING] Validation Macro-F1 stopped improving.")
            break

print(f"\nTraining complete! Best model at Epoch {best_epoch} with Val Macro-F1: {best_val_f1:.2f}%")


---
## 6. Diagnostic Learning Curves

We plot the training vs. validation loss and accuracy trajectories to examine:
1. **Convergence Stability:** Smooth decrease in weighted cross-entropy loss.
2. **Overfitting Detection:** Verifying that train and validation curves remain tightly bounded with minimal generalization gap.


In [ ]:
epochs_range = range(1, len(history["train_loss"]) + 1)
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(epochs_range, history["train_loss"], "b-o", label="Train Loss", linewidth=2)
axes[0].plot(epochs_range, history["val_loss"], "r--s", label="Validation Loss", linewidth=2)
axes[0].set_title("Training vs Validation Loss", fontsize=13, fontweight="bold")
axes[0].set_xlabel("Epoch", fontsize=11)
axes[0].set_ylabel("Weighted Cross-Entropy Loss", fontsize=11)
axes[0].legend(fontsize=11)
axes[0].grid(True, linestyle=":", alpha=0.6)

axes[1].plot(epochs_range, history["train_acc"], "b-o", label="Train Accuracy", linewidth=2)
axes[1].plot(epochs_range, history["val_acc"], "r--s", label="Val Accuracy", linewidth=2)
axes[1].plot(epochs_range, history["val_macro_f1"], "g-^", label="Val Macro-F1", linewidth=2)
axes[1].set_title("Accuracy & Macro-F1 Trajectory", fontsize=13, fontweight="bold")
axes[1].set_xlabel("Epoch", fontsize=11)
axes[1].set_ylabel("Score (%)", fontsize=11)
axes[1].legend(fontsize=11)
axes[1].grid(True, linestyle=":", alpha=0.6)

plt.tight_layout()
plt.savefig(FIGURES_DIR / "lstm_learning_curves.png", dpi=300)
plt.show()


---
## 7. Model Evaluation on Unseen Test Subjects (Volunteers 26–30)

We now load the top checkpoint weights (`best_lstm_model.pth`) and evaluate on the **unseen test subjects**.
Because subjects 26–30 were completely held out from both training and validation, these results demonstrate true real-world generalization.


In [ ]:
model.load_state_dict(torch.load(CHECKPOINT_PATH, map_location=device))
model.eval()

# GPU/MPS Warm-up iterations to avoid initial benchmark anomalies
warmup_x = torch.randn(64, 128, 6, device=device)
with torch.no_grad():
    for _ in range(5):
        _ = model(warmup_x)
synchronize_device(device)

te_preds, te_targets, te_probs, latencies = [], [], [], []

with torch.no_grad():
    for bx, by in test_loader:
        bx = bx.to(device)
        synchronize_device(device)
        t0 = time.perf_counter()
        logits = model(bx)
        synchronize_device(device)
        t1 = time.perf_counter()

        latencies.append((t1 - t0) * 1000.0 / bx.size(0))
        te_probs.extend(torch.softmax(logits, dim=1).cpu().numpy())
        te_preds.extend(torch.argmax(logits, dim=1).cpu().numpy())
        te_targets.extend(by.numpy())

te_preds = np.array(te_preds)
te_targets = np.array(te_targets)
te_probs = np.array(te_probs)

test_acc = accuracy_score(te_targets, te_preds) * 100.0
test_macro_f1 = f1_score(te_targets, te_preds, average="macro") * 100.0
test_weighted_f1 = f1_score(te_targets, te_preds, average="weighted") * 100.0
test_precision = precision_score(te_targets, te_preds, average="macro", zero_division=0) * 100.0
test_recall = recall_score(te_targets, te_preds, average="macro", zero_division=0) * 100.0
mcc = matthews_corrcoef(te_targets, te_preds) * 100.0
roc_auc = roc_auc_score(te_targets, te_probs, multi_class="ovr", average="macro") * 100.0
avg_lat = float(np.mean(latencies))

print("==================================================")
print("            FINAL UNSEEN TEST RESULTS             ")
print("==================================================")
print(f"Overall Accuracy:          {test_acc:.2f}%")
print(f"Macro-Averaged F1-Score:   {test_macro_f1:.2f}%")
print(f"Weighted F1-Score:         {test_weighted_f1:.2f}%")
print(f"Macro Precision:           {test_precision:.2f}%")
print(f"Macro Recall:              {test_recall:.2f}%")
print(f"MCC (Matthews):            {mcc:.2f}%")
print(f"ROC-AUC (Macro OvR):       {roc_auc:.2f}%")
print(f"Inference Latency/Sample:  {avg_lat:.3f} ms")
print("==================================================\n")

print("Classification Report:\n")
print(classification_report(te_targets, te_preds, target_names=CLASS_NAMES, zero_division=0))


---
## 8. Normalized 12x12 Confusion Matrix Heatmap

The Confusion Matrix illustrates:
* **Dynamic Separation:** Walking, stairs, and laying achieve near 100% precision and recall.
* **Static Ambiguity:** Sitting vs. Standing share similar vertical gravity vectors, leading to mild natural overlap.
* **Postural Transitions:** Thanks to inverse class weighting, rare transitions achieve high recall rather than being swallowed by majority classes.


In [ ]:
cm = confusion_matrix(te_targets, te_preds, labels=range(12))
cm_norm = cm.astype("float") / cm.sum(axis=1)[:, np.newaxis]

plt.figure(figsize=(12, 10))
sns.heatmap(
    cm_norm, annot=True, fmt=".1%", cmap="Blues",
    xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES,
    cbar_kws={"label": "Normalized Recall Rate"}
)
plt.title("Normalized 12-Class Confusion Matrix (ActivityLSTM)", fontsize=14, fontweight="bold", pad=15)
plt.xlabel("Predicted Activity Class", fontsize=12, fontweight="semibold")
plt.ylabel("Ground Truth Class", fontsize=12, fontweight="semibold")
plt.xticks(rotation=45, ha="right", fontsize=10)
plt.yticks(rotation=0, fontsize=10)
plt.tight_layout()
plt.savefig(FIGURES_DIR / "lstm_confusion_matrix.png", dpi=300)
plt.show()


---
## 9. Critical Analysis, Architectural Trade-offs & Viva Defense

### 1. Generalization & Zero Data Leakage:
* By enforcing a strict **subject-wise partition** (Users 1–21 for training, 22–25 for validation, and 26–30 for testing), test metrics reflect true generalization on completely unseen human subjects rather than memorizing individual gait patterns. Shuffling across overlapping windows would have introduced severe data leakage.

### 2. Empirical Performance & Per-Class Nuances:
* **Dynamic Steady-State Activities:** `WALKING`, `WALKING_UPSTAIRS`, `WALKING_DOWNSTAIRS`, and `LAYING` achieve near-perfect performance (99%–100% precision and recall).
* **Static Postural Disambiguation:** `SITTING` vs. `STANDING` exhibit mild mutual confusion (85%–89% precision/recall) because a waist-mounted sensor measures nearly identical vertical gravitational vectors in both static upright states.
* **Postural Transitions (Class Imbalance Response):** 
  - 5 out of 6 rare postural shifts achieved strong recall: `SIT_TO_STAND` (100%), `LIE_TO_STAND` (100%), `STAND_TO_SIT` (92%), `STAND_TO_LIE` (82%), and `LIE_TO_SIT` (80%).
  - `SIT_TO_LIE` achieved a lower recall of 62% due to intermediate kinematic overlap with `STAND_TO_LIE` and settling into `LAYING`. This demonstrates that while inverse class weighting substantially prevents minority starvation, distinguishing brief, subtle transitions from similar trajectories remains an inherent challenge.

### 3. Computational Cost & Real-Time Wearable Feasibility:
* **Latency:** Synchronized inference latency is approximately **0.13–0.22 ms per 2.56-second window** on modern hardware, well within the 20 ms budget of a 50 Hz sensor stream.
* **Model Footprint:** With 313,420 parameters (1.20 MB checkpoint), the model easily fits into embedded RAM of microcontrollers and smartwatches (e.g. ARM Cortex-M or Apple Watch).
* **Key Architectural Trade-Off (Causal vs. Non-Causal):** Because this network uses a **Bidirectional** LSTM, the backward pass requires the entire 128-sample window (2.56 seconds) to be buffered before inference can occur. For strict zero-delay streaming, a unidirectional LSTM or causal TCN would be required, though with a known trade-off in transitional context modeling.

---
*SE4050 Deep Learning Assignment (2026) | BSc (Hons) in Information Technology*
